In [1]:
# Imports

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
%%time

# Load prepared data

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_pickle(PROCESSED_DIR / "train_model_df.pkl")
test_df = pd.read_pickle(PROCESSED_DIR / "test_model_df.pkl")

train_df.shape, test_df.shape

CPU times: user 1.37 s, sys: 2.25 s, total: 3.62 s
Wall time: 4 s


((800000, 20), (200000, 20))

In [5]:
%%time

# Optional downsampling for faster experimentation
# Keep class distribution stable using stratified sampling.

TRAIN_SAMPLE_SIZE = 200_000

if len(train_df) > TRAIN_SAMPLE_SIZE:
    train_df_sample = (
        train_df
        .groupby("target_sentiment", group_keys=False)
        .apply(lambda x: x.sample(
            n=int(TRAIN_SAMPLE_SIZE * len(x) / len(train_df)),
            random_state=42
        ))
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )
else:
    train_df_sample = train_df.copy()

train_df_sample["target_sentiment"].value_counts(normalize=True)

CPU times: user 502 ms, sys: 93.9 ms, total: 596 ms
Wall time: 627 ms


target_sentiment
positive    0.668183
negative    0.265271
neutral     0.066545
Name: proportion, dtype: float64

In [6]:
# Define helper function for model evaluation

def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluate a fitted model and return key metrics.
    """
    preds = model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average="macro")
    weighted_f1 = f1_score(y_test, preds, average="weighted")

    print(f"Model: {model_name}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print()
    print(classification_report(y_test, preds))

    return {
        "model_name": model_name,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

In [7]:
# Define ablation experiments
# We compare whether categories, tips, and location improve text-based sentiment prediction.

experiments = {
    "review_only": {
        "text_col": "text_review_only",
        "numeric_cols": []
    },
    "review_plus_categories": {
        "text_col": "text_review_categories",
        "numeric_cols": []
    },
    "review_plus_categories_tips": {
        "text_col": "text_review_categories_tips",
        "numeric_cols": []
    },
    "full_context_with_numeric": {
        "text_col": "text_full_context",
        "numeric_cols": ["review_word_count", "review_char_count", "tip_count", "avg_tip_compliment"]
    }
}

In [8]:
%%time

# Train baseline models for each experiment
# Important: TF-IDF is fitted only on the training data inside the pipeline.

results = []
trained_models = {}

y_train = train_df_sample["target_sentiment"]
y_test = test_df["target_sentiment"]

for experiment_name, config in experiments.items():
    text_col = config["text_col"]
    numeric_cols = config["numeric_cols"]

    feature_cols = [text_col] + numeric_cols

    transformers = [
        (
            "text",
            TfidfVectorizer(
                stop_words="english",
                ngram_range=(1, 2),
                min_df=5,
                max_features=100_000
            ),
            text_col
        )
    ]

    if numeric_cols:
        transformers.append(
            (
                "numeric",
                StandardScaler(),
                numeric_cols
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                n_jobs=-1
            ))
        ]
    )

    X_train = train_df_sample[feature_cols]
    X_test = test_df[feature_cols]

    model.fit(X_train, y_train)

    metrics = evaluate_model(
        model=model,
        X_test=X_test,
        y_test=y_test,
        model_name=experiment_name
    )

    results.append(metrics)
    trained_models[experiment_name] = model

Model: review_only
Accuracy: 0.8696
Macro F1: 0.7467
Weighted F1: 0.8817

              precision    recall  f1-score   support

    negative       0.88      0.86      0.87     56153
     neutral       0.35      0.58      0.43     14355
    positive       0.97      0.90      0.94    129492

    accuracy                           0.87    200000
   macro avg       0.73      0.78      0.75    200000
weighted avg       0.90      0.87      0.88    200000

Model: review_plus_categories
Accuracy: 0.8690
Macro F1: 0.7472
Weighted F1: 0.8814

              precision    recall  f1-score   support

    negative       0.88      0.86      0.87     56153
     neutral       0.35      0.59      0.43     14355
    positive       0.97      0.90      0.94    129492

    accuracy                           0.87    200000
   macro avg       0.73      0.78      0.75    200000
weighted avg       0.90      0.87      0.88    200000

Model: review_plus_categories_tips
Accuracy: 0.8437
Macro F1: 0.7180
Weighted F

In [11]:
# Compare ablation results

results_df = (
    pd.DataFrame(results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

results_df

,model_name,accuracy,macro_f1,weighted_f1
0,review_plus_categories,0.868965,0.747216,0.881414
1,review_only,0.869580,0.746730,0.881670
2,full_context_with_numeric,0.844525,0.719202,0.859024
3,review_plus_categories_tips,0.843675,0.717998,0.858166


In [12]:
# Select best experiment

best_experiment_name = results_df.loc[0, "model_name"]
best_config = experiments[best_experiment_name]

best_experiment_name, best_config

('review_plus_categories',
 {'text_col': 'text_review_categories', 'numeric_cols': []})

In [13]:
%%time

# Grid search on the best feature setup
# Grid search is performed only on training data.

text_col = best_config["text_col"]
numeric_cols = best_config["numeric_cols"]
feature_cols = [text_col] + numeric_cols

transformers = [
    (
        "text",
        TfidfVectorizer(stop_words="english"),
        text_col
    )
]

if numeric_cols:
    transformers.append(
        (
            "numeric",
            StandardScaler(),
            numeric_cols
        )
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

param_grid = {
    "preprocessor__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessor__text__min_df": [5, 10],
    "preprocessor__text__max_features": [50_000, 100_000],
    "classifier__C": [0.5, 2.0]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(train_df_sample[feature_cols], y_train)

grid_search.best_params_, grid_search.best_score_

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] END classifier__C=0.5, preprocessor__text__max_features=50000, preprocessor__text__min_df=10, preprocessor__text__ngram_range=(1, 1); total time=  23.3s
[CV] END classifier__C=0.5, preprocessor__text__max_features=100000, preprocessor__text__min_df=5, preprocessor__text__ngram_range=(1, 1); total time=  26.0s
[CV] END classifier__C=0.5, preprocessor__text__max_features=100000, preprocessor__text__min_df=5, preprocessor__text__ngram_range=(1, 1); total time=  20.8s
[CV] END classifier__C=0.5, preprocessor__text__max_features=100000, preprocessor__text__min_df=10, preprocessor__text__ngram_range=(1, 1); total time=  21.8s
[CV] END classifier__C=2.0, preprocessor__text__max_features=50000, preprocessor__text__min_df=5, preprocessor__text__ngram_range=(1, 1); total time=  27.0s
[CV] END classifier__C=2.0, preprocessor__text__max_features=50000, preprocessor__text__min_df=10, preprocessor__text__ngram_range=(1, 1); total time

({'classifier__C': 2.0,
  'preprocessor__text__max_features': 100000,
  'preprocessor__text__min_df': 5,
  'preprocessor__text__ngram_range': (1, 2)},
 np.float64(0.7447098466979004))

In [14]:
# Evaluate best model on untouched test set

best_model = grid_search.best_estimator_

final_metrics = evaluate_model(
    model=best_model,
    X_test=test_df[feature_cols],
    y_test=y_test,
    model_name="best_grid_search_model"
)

final_metrics

Model: best_grid_search_model
Accuracy: 0.8736
Macro F1: 0.7470
Weighted F1: 0.8836

              precision    recall  f1-score   support

    negative       0.88      0.87      0.87     56153
     neutral       0.35      0.55      0.43     14355
    positive       0.97      0.91      0.94    129492

    accuracy                           0.87    200000
   macro avg       0.73      0.78      0.75    200000
weighted avg       0.90      0.87      0.88    200000



{'model_name': 'best_grid_search_model',
 'accuracy': 0.87358,
 'macro_f1': 0.7470058412247517,
 'weighted_f1': 0.8836466460341381}

In [15]:
# Confusion matrix

test_predictions = best_model.predict(test_df[feature_cols])

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=["negative", "neutral", "positive"]
)

cm_df = pd.DataFrame(
    cm,
    index=["actual_negative", "actual_neutral", "actual_positive"],
    columns=["pred_negative", "pred_neutral", "pred_positive"]
)

cm_df

,pred_negative,pred_neutral,pred_positive
actual_negative,48764,5973,1416
actual_neutral,3826,7874,2655
actual_positive,2918,8496,118078


In [16]:
%%time

# Add predictions and probabilities if available

test_output = test_df.copy()
test_output["predicted_sentiment"] = test_predictions

if hasattr(best_model.named_steps["classifier"], "predict_proba"):
    proba = best_model.predict_proba(test_df[feature_cols])
    classes = best_model.named_steps["classifier"].classes_

    for idx, class_name in enumerate(classes):
        test_output[f"prob_{class_name}"] = proba[:, idx]

test_output.head()

CPU times: user 11.8 s, sys: 310 ms, total: 12.1 s
Wall time: 13 s


,review_id,business_id,user_id,review_date,review_stars,target_sentiment,review_text,text_review_only,text_review_categories,text_review_categories_tips,...,categories,sample_tips,review_word_count,review_char_count,tip_count,avg_tip_compliment,predicted_sentiment,prob_negative,prob_neutral,prob_positive
0,zNfIWVHthsMn-Qyrw-JfGg,6KjQQDmLqazKCq-l2Y_Lnw,oJ_72Qj8itPVhYE4yCIrrA,2021-09-14 16:30:38,1.0,negative,The customer service here was terrible. There ...,The customer service here was terrible. There ...,Review: The customer service here was terrible...,Review: The customer service here was terrible...,...,"American (Traditional), Fast Food, Food, Resta...",Super Sub. | I love this place. I wish the guy...,55,286,4,0.000000,negative,0.997314,0.002681,0.000005
1,sVsf2nIKjWCo3ViVprJcPQ,CFxs7aHPs60lgQmSvAkmPA,Hf3328ygirIBIzlJrA_3Ag,2021-09-14 16:30:50,1.0,negative,EDITED (9/14/2021) TO ADD: Please note the of...,EDITED (9/14/2021) TO ADD: Please note the of...,Review: EDITED (9/14/2021) TO ADD: Please not...,Review: EDITED (9/14/2021) TO ADD: Please not...,...,"Health & Medical, General Dentistry, Pediatric...",Outstanding dentist. | Beware. This place is ...,287,1663,7,0.142857,negative,0.835311,0.159307,0.005382
2,j9gt7aQ0JDGvB7phNIfXfA,rIGX4yiwBwI9l2XC5F0dbw,yZDkdE9YNim2jqHD3WHTLA,2021-09-14 16:31:45,4.0,positive,I love supporting independent bookstores and t...,I love supporting independent bookstores and t...,Review: I love supporting independent bookstor...,Review: I love supporting independent bookstor...,...,"Books, Mags, Music & Video, Bookstores, Shopping",Spotty customer service. Missed out on a big p...,60,343,15,0.000000,positive,0.010040,0.061412,0.928548
3,aRjdrHcuKCJzgUlruh6ikw,efou6IrwQhHAXZ0e0v0Gkw,JH6S19ei4yo9g_c6M25D1A,2021-09-14 16:32:00,5.0,positive,Friendly service and outstanding flavor and qu...,Friendly service and outstanding flavor and qu...,Review: Friendly service and outstanding flavo...,Review: Friendly service and outstanding flavo...,...,"Greek, Restaurants, Sports Clubs, Arts & Enter...",Friendly service and outstanding flavor and qu...,41,246,2,0.000000,positive,0.054492,0.123398,0.822110
4,0c-YJK44XuSmWxn5SsmtoQ,utJRw1T1_zjBuCIFFIbe-w,77Hzs5MnUIRmSPbkfvsKtQ,2021-09-14 16:32:02,1.0,negative,The worst process I've ever encountered for ma...,The worst process I've ever encountered for ma...,Review: The worst process I've ever encountere...,Review: The worst process I've ever encountere...,...,"Breakfast & Brunch, Bars, American (New), Nigh...",Yummy wine! | Wednesday night live | Pizza is ...,106,591,48,0.000000,negative,0.992482,0.007431,0.000087


In [17]:
# Error analysis

error_df = test_output[
    test_output["target_sentiment"] != test_output["predicted_sentiment"]
].copy()

error_sample = error_df[
    [
        "review_text",
        "target_sentiment",
        "predicted_sentiment",
        "review_stars",
        "categories",
        "city"
    ]
].sample(20, random_state=42)

error_sample

,review_text,target_sentiment,predicted_sentiment,review_stars,categories,city
96440,Little dirty and unorganized and overpriced.Ev...,positive,neutral,4.0,"Bakeries, Restaurants, Food, Juice Bars & Smoo...",Maryland Heights
199127,I would probably give this place 1.5 stars. We...,negative,neutral,2.0,"Chicken Wings, Restaurants, Nightlife, Barbequ...",New Orleans
58909,Popped in here on a lark as we were waiting on...,neutral,positive,3.0,"Arts & Entertainment, Wineries, Wine Tasting R...",Santa Barbara
18946,I'm giving one extra star because the food is ...,negative,neutral,2.0,"Restaurants, Creperies, Cafes, Sandwiches, Bre...",Nashville
167102,The food had a good portion but sadly it had n...,neutral,negative,3.0,"Restaurants, Chinese",Tucson
4488,"Had lunch there with a friend today, was very ...",negative,neutral,2.0,"Restaurants, American (New), Nightlife, Breakf...",Mullica Hill
187845,So let's start off I really wanted to give the...,positive,neutral,4.0,"Steakhouses, Seafood, Restaurants, Desserts, A...",Tampa
81194,"Based on rave reviews from friends in Phoenix,...",positive,neutral,5.0,"Salad, Restaurants, Vegetarian, Breakfast & Br...",Tucson
2601,Some of the top Drs and staff in the area. Alw...,positive,neutral,4.0,"Pets, Veterinarians",Tampa
197252,I'm The shrimp and grits were not even enough ...,negative,neutral,2.0,"Southern, Event Planning & Services, Restauran...",New Orleans


In [18]:
%%time

# Feature importance from logistic regression
# This works for text-only or text-dominant models.

text_vectorizer = best_model.named_steps["preprocessor"].named_transformers_["text"]
classifier = best_model.named_steps["classifier"]

feature_names = text_vectorizer.get_feature_names_out()
classes = classifier.classes_

importance_tables = {}

for class_idx, class_name in enumerate(classes):
    coefs = classifier.coef_[class_idx]
    top_idx = np.argsort(coefs)[-30:][::-1]

    importance_tables[class_name] = pd.DataFrame({
        "term": feature_names[top_idx],
        "coefficient": coefs[top_idx]
    })

importance_tables["negative"].head(20)

CPU times: user 81.5 ms, sys: 1.4 ms, total: 83 ms
Wall time: 81.9 ms


,term,coefficient
0,worst,11.449954
1,terrible,8.204066
2,horrible,8.129811
3,rude,8.055437
4,awful,6.605889
5,poor,6.199925
6,disgusting,6.194443
7,bland,5.595353
8,unprofessional,5.506810
9,zero,5.491829


In [19]:
%%time

# Save model outputs

import joblib

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, MODEL_DIR / "sentiment_model.pkl")

test_output.to_pickle(PROCESSED_DIR / "test_predictions.pkl")
results_df.to_csv(PROCESSED_DIR / "ablation_results.csv", index=False)

print("Model and outputs saved.")

Model and outputs saved.
CPU times: user 1.03 s, sys: 377 ms, total: 1.41 s
Wall time: 1.94 s
